# Singular Value Decomposition (SVD): image compression and denoising

In this notebook we study two closely related applications of the singular value decomposition (SVD) using a grayscale image:

1. **Image compression**: approximate the image by a low-rank matrix and store only the most important singular directions.
2. **Image denoising**: add Gaussian noise to the image and then use a truncated SVD to recover a cleaner approximation.

These two tasks are based on the same idea: many natural images can be approximated well by keeping only the dominant singular values and singular vectors. In compression, this reduces storage. In denoising, it suppresses weak components that are often associated with noise.


## Learning goals

By the end of this notebook, you should be able to:

- compute the economy SVD of an image matrix,
- build low-rank approximations of an image,
- interpret singular values as a hierarchy of importance,
- measure reconstruction quality with Frobenius-norm errors,
- estimate storage savings from truncated SVD,
- explain how truncated SVD can remove part of the noise in an image,
- discuss the trade-off between compression, denoising, and image quality.


## 1. Load and visualize the image

We will work with a grayscale image stored as a matrix $X\in\mathbb{R}^{n\times m}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

plt.rcParams["figure.figsize"] = (6, 6)

#image_path = "data/Image_Owl.png"
image_path = "data/Image_Cheetah.png"
X = np.array(Image.open(image_path).convert("L"), dtype=float)
n, m = X.shape

print(f"Image shape: {X.shape}")
print(f"Pixel range: [{X.min():.1f}, {X.max():.1f}]")

plt.imshow(X, cmap="gray")
plt.title("Original grayscale image")
plt.axis("off")
plt.show()

## 2. Full SVD and economy SVD

Recall that the full SVD of the image matrix is
$$
X = U\Sigma V^T.
$$
If $X\in\mathbb{R}^{n\times m}$ with $n\ge m$, the **economy SVD** keeps only the nonzero rectangular part and is often more efficient in practice.

In this notebook we will use both forms:
- the **full SVD** to inspect the matrix shapes,
- the **economy SVD** to build low-rank approximations efficiently.

In [ ]:
# TODO: compute the full SVD
# U_full, s_full, VT_full = ...

# TODO: compute the economy SVD
# U, s, VT = ...

# print("Full SVD shapes:", U_full.shape, s_full.shape, VT_full.shape)
# print("Economy SVD shapes:", U.shape, s.shape, VT.shape)

## 3. Singular values and cumulative information

The singular values are ordered from largest to smallest. If they decay rapidly, the image is well approximated by a low-rank matrix.

We will plot:
- the singular values on a semilog scale,
- the cumulative sum of singular values,
- the cumulative sum of squared singular values.

The squared singular values are especially important because
$$
\|X\|_F^2 = \sum_{k} \sigma_k^2.
$$

In [ ]:
# TODO: plot the singular values and the two cumulative curves
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# ...
# plt.tight_layout()
# plt.show()

## 4. Low-rank image reconstruction

The rank-$r$ truncated SVD approximation is
$$
\widetilde X_r = U_r\Sigma_r V_r^T.
$$
We will reconstruct the image for a few values of $r$ and compare the visual quality.

For compression, storing a rank-$r$ approximation requires
$$
 nr + r + mr
$$
real numbers instead of $nm$.

In [ ]:
r_values = [5, 20, 50, 100]

# TODO: create a 2x2 grid with the rank-r reconstructions
# For each r:
#   Ur = ...
#   Sr = ...
#   VTr = ...
#   Xr = ...
#   storage_fraction = ...
#   plot the image and show the storage percentage in the title

## 5. $U_r^T U_r$ versus $U_r U_r^T$

Let $U_r$ be the matrix containing the first $r$ left singular vectors.

Because its columns are orthonormal, we expect
$$
U_r^T U_r = I_r.
$$
However, unless $r=n$, we do **not** expect
$$
U_r U_r^T = I_n.
$$

We will verify both facts numerically.

In [ ]:
r_test = 40

# TODO: extract U_r for r = r_test
# Ur = ...

# TODO: build the two products
# left_product = ...
# right_product = ...

# TODO: print the norms
# print("||U_r^T U_r - I_r||_F =", ...)
# print("||U_r U_r^T - I_n||_F =", ...)

## 6. Reconstruction error and captured variance

For the rank-$r$ approximation $\widetilde X_r$, define the **relative Frobenius reconstruction error** by
$$
\frac{\|X-\widetilde X_r\|_F}{\|X\|_F}.
$$
Its square is
$$
\frac{\|X-\widetilde X_r\|_F^2}{\|X\|_F^2},
$$
which may be interpreted as the fraction of **missing variance**.

We will compare three quantities as functions of $r$:
- relative Frobenius error,
- captured variance $= 1 -$ missing variance,
- cumulative sum of singular values.

In [ ]:
# TODO: compute the curves as functions of rank
# Hint: use cumulative sums of s and s**2

# ranks = np.arange(1, len(s) + 1)
# rel_fro_error = ...
# missing_variance = ...
# captured_variance = ...
# cumulative_sigma = ...

# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# ...
# plt.tight_layout()
# plt.show()

In [ ]:
# TODO: find the first rank where each quantity reaches 99%
# - captured variance >= 0.99
# - 1 - relative Frobenius error >= 0.99
# - cumulative sum of singular values >= 0.99

# r_var_99 = ...
# r_fro_99 = ...
# r_sigma_99 = ...

# print("Rank for 99% captured variance:", r_var_99)
# print("Rank for 99% of 1 - relative Frobenius error:", r_fro_99)
# print("Rank for 99% cumulative singular-value sum:", r_sigma_99)

## 7. Add Gaussian noise to the image

We now extend the notebook from **compression** to **denoising**.

Starting from the clean grayscale owl image, we add Gaussian noise to obtain a corrupted image. Then we compute the SVD of the noisy image and compare two truncation strategies:

- an **SVD hard threshold** based on the known noise level,
- a **90% cumulative-sum cutoff**.

This lets us compare visual quality, truncation rank, and reconstruction error.


In [ ]:
# Create a noisy version of the image
np.random.seed(7)

noise_std = 30.0
X_noisy = X + noise_std * np.random.randn(*X.shape)
X_noisy = np.clip(X_noisy, 0, 255)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(X, cmap="gray")
axes[0].set_title("Clean image")
axes[0].axis("off")

axes[1].imshow(X_noisy, cmap="gray")
axes[1].set_title("Noisy image")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## 8. Compute the SVD of the noisy image

Let
$$
X_{\text{noisy}} = U\Sigma V^T.
$$

We will inspect the singular values and then choose truncation ranks in two different ways.


In [ ]:
# TODO: compute the economy SVD of the noisy image

# U_noisy, s_noisy, VT_noisy = ...


## 9. Choose truncation ranks

We now study two truncation ideas:

1. **Hard threshold** based on the known Gaussian noise level.
2. **90% cumulative-sum cutoff**.

Because the image matrix is rectangular, we use the rectangular hard-threshold formula
$$
\tau = \lambda(\beta)\sqrt{n}\,\sigma,
$$
where $\beta$ is the aspect ratio and $\sigma$ is the noise standard deviation.


In [ ]:
# TODO: compute the hard threshold and the 90% cumulative-sum cutoff

n_rows, n_cols = X_noisy.shape
beta = min(n_rows, n_cols) / max(n_rows, n_cols)

# lambda_beta = ...
# cutoff = ...

# r_hard = ...
# cumulative_sum = ...
# r90 = ...

print(f"hard-threshold rank = {r_hard}")
print(f"90% cumulative-sum rank = {r90}")


In [ ]:
# TODO: plot the singular values and the cumulative sum

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: singular values and hard threshold
# axes[0].semilogy(...)
# axes[0].axhline(...)
# axes[0].set_title(...)

# Right: cumulative sum and 90% cutoff
# axes[1].plot(...)
# axes[1].axvline(...)
# axes[1].set_title(...)

plt.tight_layout()
plt.show()


## 10. Reconstruct denoised images

Using the two ranks above, build two low-rank approximations of the noisy image:

- one from the **hard threshold**,
- one from the **90% cumulative-sum cutoff**.

Compare them with the clean and noisy images.


In [ ]:
# TODO: reconstruct denoised images using both truncation rules

# X_hard = ...
# X_90 = ...

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(X, cmap="gray")
axes[0].set_title("Clean")
axes[0].axis("off")

axes[1].imshow(X_noisy, cmap="gray")
axes[1].set_title("Noisy")
axes[1].axis("off")

axes[2].imshow(X_hard, cmap="gray")
axes[2].set_title(f"Hard threshold (r={r_hard})")
axes[2].axis("off")

axes[3].imshow(X_90, cmap="gray")
axes[3].set_title(f"90% cutoff (r={r90})")
axes[3].axis("off")

plt.tight_layout()
plt.show()


## 11. Compare the denoising results quantitatively

We compare the relative Frobenius errors with respect to the clean image:
$$
\frac{\|X-X_{\text{approx}}\|_F}{\|X\|_F}.
$$

This helps us check whether the denoised images are genuinely closer to the clean image than the noisy one.


In [ ]:
# Relative Frobenius errors
err_noisy = np.linalg.norm(X - X_noisy, ord="fro") / np.linalg.norm(X, ord="fro")

# TODO: compute the errors of the two denoised images
# err_hard = ...
# err_90 = ...

print(f"Noisy image error:            {err_noisy:.4f}")
print(f"Hard-threshold SVD error:     {err_hard:.4f}")
print(f"90% cumulative-sum error:     {err_90:.4f}")


## Final discussion

1. Why do the first singular values capture most of the visible structure of the image?
2. How does increasing the truncation rank improve the compressed image?
3. Why does truncated SVD reduce storage requirements?
4. Why can truncated SVD remove part of the Gaussian noise?
5. What is the idea behind the SVD hard threshold?
6. Why can the 90% cumulative-sum cutoff keep more noise than the hard threshold?
7. How are compression and denoising based on the same low-rank idea?
